In [2]:

import nest_asyncio
nest_asyncio.apply()  # 중첩 이벤트 루프 허용
import os
import json
import logging
import asyncio
import pandas as pd
import numpy as np
import openai
from datetime import datetime
import sys
from typing import List, Dict, Optional
# from tenacity import retry, stop_after_attempt, wait_exponential
import re
import gc
import matplotlib.pyplot as plt
from tqdm import tqdm
api = pd.read_csv('../../data/info.csv')
api_key = api.loc[1][1]

# 한글 폰트 설정
import matplotlib.font_manager as fm
import matplotlib as mpl

# 한글 폰트 경로 설정 (맥OS 기준)
font_path = '/System/Library/Fonts/AppleSDGothicNeo.ttc'  # 맥OS의 기본 한글 폰트
font_prop = fm.FontProperties(fname=font_path)

# matplotlib 기본 폰트 설정
plt.rc('font', family=font_prop.get_name())
mpl.rcParams['axes.unicode_minus'] = False  # 마이너스 기호 깨짐 방지

# 폰트 확인
print(f"설정된 폰트: {font_prop.get_name()}")
print(f"사용 가능한 한글 폰트:")
for font in fm.findSystemFonts():
    if 'gothic' in font.lower() or 'gulim' in font.lower() or 'malgun' in font.lower() or 'batang' in font.lower():
        print(f" - {font}")

from scipy.spatial.distance import cosine
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity
from scipy.special import softmax
import numpy as np
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
from scipy.spatial.distance import cosine

df = pd.read_hdf('/Users/nam-yeong/git/prj_centum/gpt_word/final_result/preprocessed_final_df.h5', 
                 key='df')

/var/folders/7z/944bgjcs659fsp58nc954qgm0000gn/T/ipykernel_2788/3294892761.py:19: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  api_key = api.loc[1][1]


설정된 폰트: Apple SD Gothic Neo
사용 가능한 한글 폰트:
 - /System/Library/Fonts/AppleSDGothicNeo.ttc
 - /System/Library/Fonts/Supplemental/NotoSansGothic-Regular.ttf
 - /System/Library/Fonts/Supplemental/AppleGothic.ttf


In [45]:
# pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', None)
# pd.set_option('display.max_colwidth', 100)
# pd.set_option('display.max_colwidth', None)

### 임베딩

#### 환자 시계열 요약 함수

In [3]:
def create_patient_timeline_summary(patient_visits, max_tokens=4000):
    """환자의 전체 방문 데이터에서 토큰 제한 내 요약 생성"""
    # 방문 정렬
    patient_visits = patient_visits.sort_values('날짜')
    patient_id = patient_visits['환자번호'].iloc[0]
    
    # 치료 기간 요약
    first_visit_date = patient_visits['날짜'].iloc[0]
    last_visit_date = patient_visits['날짜'].iloc[-1]
    
    if hasattr(first_visit_date, 'strftime'):
        first_visit_str = first_visit_date.strftime('%Y-%m-%d')
        last_visit_str = last_visit_date.strftime('%Y-%m-%d')
    else:
        first_visit_str = str(first_visit_date)
        last_visit_str = str(last_visit_date)
    
    try:
        days_elapsed = (last_visit_date - first_visit_date).days
    except:
        days_elapsed = 0
    
    visit_count = len(patient_visits)
    
    # 핵심 임상 지표 변화 계산
    key_metrics = {
        'CC_vas': '통증강도(VAS)', 
        'CMO_before': '편안한 개구량(CMO)', 
        'MMO_before': '최대 개구량(MMO)',
        'dif_MMO_CMO': 'MMO-CMO 차이'
    }
    
    changes = {}
    for metric, desc in key_metrics.items():
        if metric in patient_visits.columns:
            # 첫 방문과 마지막 방문 값
            metric_data = patient_visits[metric].dropna()
            if len(metric_data) >= 2:
                first_val = metric_data.iloc[0]
                last_val = metric_data.iloc[-1]
                
                # 변화량 및 변화율 계산
                abs_change = last_val - first_val
                pct_change = (abs_change / first_val * 100) if first_val != 0 else float('inf')
                
                # 변화 패턴 분석 (선형/비선형)
                if len(metric_data) >= 3:
                    # 간단한 선형 추세 분석
                    import numpy as np
                    x = np.arange(len(metric_data))
                    y = metric_data.values
                    from scipy import stats
                    slope, intercept, r_value, p_value, std_err = stats.linregress(x, y)
                    
                    # 변화 패턴 분류
                    if abs(r_value) > 0.8:
                        pattern = "선형" if r_value > 0 else "선형 감소"
                    else:
                        # 변화 방향의 일관성 확인
                        diffs = np.diff(y)
                        direction_changes = sum(1 for i in range(len(diffs)-1) if diffs[i] * diffs[i+1] < 0)
                        pattern = "변동적" if direction_changes > 0 else "일관적"
                else:
                    pattern = "정보 부족"
                
                changes[metric] = {
                    'first': first_val,
                    'last': last_val,
                    'abs_change': abs_change,
                    'pct_change': pct_change,
                    'pattern': pattern
                }
    
    # 증상 변화 파악
    symptom_cols = ['Noise_Code', 'CC_pain_type', '습관_habit_type', 'CC_location']
    symptom_changes = {}
    
    for symptom in symptom_cols:
        if symptom in patient_visits.columns:
            values = patient_visits[symptom].dropna()
            if len(values) >= 2:
                first_val = values.iloc[0]
                last_val = values.iloc[-1]
                
                # 실제 값이 있고 Unknown이 아닌 경우만 처리
                if pd.notna(first_val) and pd.notna(last_val) and first_val != 'Unknown' and first_val != '':
                    # 증상 해소 여부 판단
                    resolved = (last_val == 'Unknown' or last_val == '' or 
                               (symptom == 'Noise_Code' and last_val == 'No-Noise'))
                    
                    symptom_changes[symptom] = {
                        'first': first_val,
                        'last': last_val,
                        'resolved': resolved,
                        'changed': first_val != last_val
                    }
    
    # 치료 방법 추출
    treatment_cols = ['약_medication_type', '장치_device_type', '찜질_status', '마사지, 스트레칭_type']
    treatments = {}
    
    for col in treatment_cols:
        if col in patient_visits.columns:
            values = patient_visits[col].dropna()
            if len(values) > 0:
                treatments[col] = values.iloc[-1] if pd.notna(values.iloc[-1]) and values.iloc[-1] != 'Unknown' else None
    
    # 요약 생성
    summary_parts = []
    
    # 1. 기본 정보
    basic_info = f"""환자 {patient_id} 시계열 요약:
- 치료 기간: {days_elapsed}일 (방문 {visit_count}회)
- 첫 방문: {first_visit_str}
- 마지막 방문: {last_visit_str}"""
    summary_parts.append(basic_info)
    
    # 2. 핵심 지표 변화
    if changes:
        metrics_summary = "핵심 지표 변화:"
        for metric, data in changes.items():
            description = key_metrics[metric]
            direction = "감소" if data['abs_change'] < 0 else "증가"
            
            if metric == 'CC_vas':  # 통증은 감소가 개선
                is_improvement = data['abs_change'] < 0
            else:  # 개구량은 증가가 개선
                is_improvement = data['abs_change'] > 0
            
            improvement_text = "개선" if is_improvement else "악화"
            
            metrics_summary += f"\n- {description}: {data['first']:.1f} → {data['last']:.1f} ({abs(data['abs_change']):.1f} {direction}, {abs(data['pct_change']):.1f}%, {improvement_text})"
            if data['pattern'] != "정보 부족":
                metrics_summary += f" [{data['pattern']} 패턴]"
        
        summary_parts.append(metrics_summary)
    
    # 3. 증상 변화
    if symptom_changes:
        symptoms_summary = "증상 변화:"
        for symptom, data in symptom_changes.items():
            if data['resolved']:
                status = "소실됨"
            elif data['changed']:
                status = f"{data['first']} → {data['last']} (변화)"
            else:
                status = f"{data['first']} (유지)"
            
            symptoms_summary += f"\n- {symptom}: {status}"
        
        summary_parts.append(symptoms_summary)
    
    # 4. 치료 방법
    if treatments:
        treatments_summary = "적용된 치료:"
        for col, value in treatments.items():
            if value:
                if col == '약_medication_type':
                    treatments_summary += f"\n- 약물 치료: {value}"
                elif col == '장치_device_type':
                    treatments_summary += f"\n- 장치 치료: {value}"
                elif col == '찜질_status' and value == 1:
                    treatments_summary += f"\n- 찜질 요법 적용"
                elif col == '마사지, 스트레칭_type':
                    treatments_summary += f"\n- 마사지/스트레칭: {value}"
        
        summary_parts.append(treatments_summary)
    
    # 최종 요약 결합
    complete_summary = "\n\n".join(summary_parts)
    
    # 토큰 수 예상 및 제한
    estimated_tokens = len(complete_summary.split()) / 0.75
    
    if estimated_tokens > max_tokens:
        # 토큰 제한 초과 시 핵심 정보만 유지
        reduced_summary = f"""환자 {patient_id} 핵심 요약:
- 치료: {days_elapsed}일, {visit_count}회 방문
- 첫/마지막: {first_visit_str}/{last_visit_str}"""
        
        # 핵심 지표 변화 간소화
        if changes:
            reduced_summary += "\n\n주요 변화:"
            for metric, data in list(changes.items())[:3]:  # 상위 3개만
                description = key_metrics[metric]
                reduced_summary += f"\n- {description}: {data['first']:.1f}→{data['last']:.1f} ({abs(data['pct_change']):.1f}%)"
        
        return reduced_summary
    
    return complete_summary

#### 현재 상태 요약 함수

In [4]:
def create_current_status_summary(last_visit):
    """환자의 마지막 방문 데이터를 기반으로 현재 상태 요약 생성"""
    summary_parts = []
    
    # 1. 기본 정보
    patient_id = last_visit['환자번호']
    visit_date = last_visit['날짜']
    
    if hasattr(visit_date, 'strftime'):
        visit_date = visit_date.strftime('%Y-%m-%d')
        
    basic_info = f"환자 {patient_id} 현재 상태 ({visit_date}):"
    summary_parts.append(basic_info)
    
    # 2. 핵심 임상 지표
    key_metrics = {
        'CC_vas': '통증강도(VAS)', 
        'CMO_before': '편안한 개구량(CMO)', 
        'MMO_before': '최대 개구량(MMO)',
        'dif_MMO_CMO': 'MMO-CMO 차이'
    }
    
    metrics_summary = "임상 지표:"
    has_metrics = False
    
    for metric, desc in key_metrics.items():
        if metric in last_visit and pd.notna(last_visit[metric]):
            metrics_summary += f"\n- {desc}: {last_visit[metric]}"
            has_metrics = True
    
    if has_metrics:
        summary_parts.append(metrics_summary)
    
    # 3. 현재 증상
    symptom_cols = ['Noise_Code', 'CC_pain_type', '습관_habit_type', 'CC_location']
    symptoms_summary = "현재 증상:"
    has_symptoms = False
    
    for symptom in symptom_cols:
        if symptom in last_visit and pd.notna(last_visit[symptom]) and last_visit[symptom] != 'Unknown' and last_visit[symptom] != '':
            symptoms_summary += f"\n- {symptom}: {last_visit[symptom]}"
            has_symptoms = True
    
    if has_symptoms:
        summary_parts.append(symptoms_summary)
    
    # 4. 압흔 및 관절음 상태
    clinical_cols = ['Tongue_ridging_Intensity', 'Mucosal_ridging_Intensity', 'Noise_Intensity', 'Noise_Direction']
    clinical_summary = "임상 징후:"
    has_clinical = False
    
    for col in clinical_cols:
        if col in last_visit and pd.notna(last_visit[col]) and last_visit[col] != 0:
            clinical_summary += f"\n- {col}: {last_visit[col]}"
            has_clinical = True
    
    if has_clinical:
        summary_parts.append(clinical_summary)
    
    # 최종 요약 결합
    current_status = "\n\n".join(summary_parts)
    return current_status

### 혼합 임베딩 생성 함수

In [5]:
def create_hybrid_embedding(patient_id, df, api_key):
    """환자의 시계열 요약과 현재 상태를 결합한 혼합 임베딩 생성"""
    # 환자 데이터 추출
    patient_visits = df[df['환자번호'] == patient_id].sort_values('날짜')
    
    if len(patient_visits) == 0:
        print(f"환자 ID {patient_id}에 대한 데이터가 없습니다.")
        return None
    
    # 환자의 마지막 방문 데이터
    last_visit = patient_visits.iloc[-1]
    
    # 1. 시계열 요약 생성
    timeline_summary = create_patient_timeline_summary(patient_visits)
    
    # 2. 현재 상태 요약 생성
    current_status = create_current_status_summary(last_visit)
    
    # 3. 두 요약을 결합
    combined_text = f"{timeline_summary}\n\n{current_status}"
    
    # 4. 임베딩 생성
    try:
        from openai import OpenAI
        client = OpenAI(api_key=api_key)
        
        response = client.embeddings.create(
            model="text-embedding-ada-002",
            input=combined_text
        )
        
        embedding = response.data[0].embedding
        
        return {
            'patient_id': patient_id,
            'embedding': embedding,
            'timeline_summary': timeline_summary,
            'current_status': current_status,
            'combined_text': combined_text
        }
    
    except Exception as e:
        print(f"환자 {patient_id}의 임베딩 생성 중 오류 발생: {str(e)}")
        return None

### 환자 단위 임베딩 생성 함수

In [6]:
def create_patient_level_embeddings(df, api_key, batch_size=20):
    """환자 단위로 혼합 임베딩 생성 (배치 처리)"""
    from tqdm import tqdm
    
    # 환자 목록 추출
    patient_ids = df['환자번호'].unique()
    total_patients = len(patient_ids)
    
    print(f"총 {total_patients}명의 환자에 대한 임베딩을 생성합니다...")
    
    # 결과 저장용 리스트
    patient_embeddings = []
    
    # 배치 단위로 처리
    for i in tqdm(range(0, total_patients, batch_size), desc="환자 임베딩 생성 중"):
        # 현재 배치의 환자 목록
        batch_patients = patient_ids[i:min(i+batch_size, total_patients)]
        
        batch_embeddings = []
        
        # 각 환자에 대해 임베딩 생성
        for patient_id in batch_patients:
            try:
                # 혼합 임베딩 생성
                embedding_result = create_hybrid_embedding(patient_id, df, api_key)
                
                if embedding_result:
                    batch_embeddings.append(embedding_result)
                
            except Exception as e:
                print(f"환자 {patient_id} 처리 중 오류 발생: {str(e)}")
        
        # 배치 임베딩 저장
        patient_embeddings.extend(batch_embeddings)
        
        # 중간 결과 저장
        batch_filename = f'patient_embeddings_batch_{i//batch_size+1}.json'
        save_embeddings_to_json(batch_embeddings, batch_filename)
        print(f"배치 {i//batch_size+1}: {len(batch_embeddings)}명 환자의 임베딩이 '{batch_filename}'에 저장되었습니다.")
    
    return patient_embeddings

#### 임베딩 JSON 저장 함수

In [4]:
def save_embeddings_to_json(embeddings, filename):
    """임베딩을 JSON 파일로 저장하는 함수"""
    import json
    import numpy as np
    from datetime import datetime
    
    # 저장 경로 설정 (기본 경로에 embeddings 폴더)
    import os
    save_dir = '/Users/nam-yeong/git/prj_centum/gpt_word/final_result/embeddings'
    os.makedirs(save_dir, exist_ok=True)
    filepath = os.path.join(save_dir, filename)
    
    # JSON 직렬화를 위해 NumPy 배열과 Timestamp 객체 처리
    serializable_embeddings = []
    for item in embeddings:
        serializable_item = item.copy()
        
        # 임베딩 벡터가 NumPy 배열인 경우 리스트로 변환
        if 'embedding' in serializable_item and hasattr(serializable_item['embedding'], 'tolist'):
            serializable_item['embedding'] = serializable_item['embedding'].tolist()
        
        # Timestamp 객체가 있는 경우 문자열로 변환
        if 'visit_date' in serializable_item and hasattr(serializable_item['visit_date'], 'strftime'):
            serializable_item['visit_date'] = serializable_item['visit_date'].strftime('%Y-%m-%d')
        
        serializable_embeddings.append(serializable_item)
    
    # 커스텀 JSON 인코더 사용
    class CustomJSONEncoder(json.JSONEncoder):
        def default(self, obj):
            # pandas Timestamp 객체 처리
            if hasattr(obj, 'strftime'):
                return obj.strftime('%Y-%m-%d')
            # NumPy 배열 처리
            if hasattr(obj, 'tolist'):
                return obj.tolist()
            return super().default(obj)
    
    try:
        with open(filepath, 'w', encoding='utf-8') as f:
            json.dump(serializable_embeddings, f, cls=CustomJSONEncoder, ensure_ascii=False, indent=2)
        return True
    except Exception as e:
        print(f"임베딩 저장 중 오류 발생: {e}")
        # 오류 시 백업 저장 시도
        backup_filepath = os.path.join(save_dir, f"backup_{datetime.now().strftime('%Y%m%d_%H%M%S')}_{filename}")
        try:
            with open(backup_filepath, 'w', encoding='utf-8') as f:
                json.dump(serializable_embeddings, f, cls=CustomJSONEncoder, ensure_ascii=False, indent=2)
            print(f"백업 파일에 저장 성공: {backup_filepath}")
            return True
        except:
            print("백업 저장도 실패했습니다.")
            return False

#### 임베딩 파일들 호출 함수

In [13]:
def load_patient_embeddings(embedding_files):
    """저장된 환자 임베딩 파일들을 로드하는 함수"""
    import json
    import os
    import numpy as np
    
    # 결과 저장을 위한 리스트
    all_embeddings = []
    
    # 각 파일 처리
    for file_path in embedding_files:
        if not os.path.exists(file_path):
            print(f"경고: {file_path} 파일이 존재하지 않습니다.")
            continue
            
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                data = json.load(f)
            
            if isinstance(data, list):
                for item in data:
                    if 'patient_id' in item and 'embedding' in item:
                        # 문자열이나 리스트로 저장된 임베딩을 NumPy 배열로 변환
                        if isinstance(item['embedding'], list):
                            item['embedding'] = np.array(item['embedding'])
                
                print(f"{file_path}에서 {len(data)}개 임베딩 로드 완료")
                all_embeddings.extend(data)
            else:
                print(f"경고: {file_path}의 형식이 예상과 다릅니다.")
                
        except Exception as e:
            print(f"{file_path} 로드 중 오류 발생: {e}")
    
    print(f"총 {len(all_embeddings)}개 환자 임베딩 로드 완료")
    return all_embeddings

#### 토큰 사이즈 에러 체크 함수

In [6]:
def check_token_size_errors(df, sample_size=10):
    """간단한 토큰 사이즈 에러 감지 함수"""
    from tqdm import tqdm
    
    # 샘플 데이터만 사용
    df_sample = df.sample(min(sample_size, len(df)), random_state=42)
    
    print(f"샘플 {len(df_sample)}개 방문 데이터에 대한 텍스트 길이 확인 중...")
    
    # 결과 저장
    text_lengths = []
    
    # 각 샘플에 대해 텍스트 생성 및 길이 확인
    for idx, row in tqdm(df_sample.iterrows()):
        try:
            # 텍스트 생성
            visit_text = create_optimized_visit_text(row)
            
            # 텍스트 길이 저장
            text_lengths.append({
                'patient_id': row['환자번호'],
                'visit_date': row['날짜'],
                'text_length': len(visit_text),
                'word_count': len(visit_text.split()),
                'estimated_tokens': len(visit_text.split()) / 0.75  # 간단한 토큰 수 추정
            })
            
        except Exception as e:
            print(f"환자 {row['환자번호']}, 방문일 {row['날짜']}: 텍스트 생성 실패 - {e}")
    
    # 간단한 분석
    if text_lengths:
        df_lengths = pd.DataFrame(text_lengths)
        
        print("\n텍스트 길이 분석 결과:")
        print(f"- 평균 텍스트 길이: {df_lengths['text_length'].mean():.1f} 문자")
        print(f"- 최대 텍스트 길이: {df_lengths['text_length'].max()} 문자")
        print(f"- 최소 텍스트 길이: {df_lengths['text_length'].min()} 문자")
        print(f"- 평균 단어 수: {df_lengths['word_count'].mean():.1f} 단어")
        print(f"- 추정 평균 토큰 수: {df_lengths['estimated_tokens'].mean():.1f} 토큰")
        
        # 임베딩 모델의 토큰 제한 (8191)을 초과할 가능성이 있는 경우 경고
        max_tokens = df_lengths['estimated_tokens'].max()
        if max_tokens > 6000:  # 8191보다 여유 있게 설정
            print(f"\n⚠️ 주의: 최대 추정 토큰 수가 {max_tokens:.1f}로, 임베딩 모델의 제한(8191)에 근접하거나 초과할 수 있습니다.")
            
            # 가장 큰 데이터 표시
            largest = df_lengths.nlargest(3, 'estimated_tokens')
            print("\n토큰 수가 많은 방문 데이터:")
            for i, (idx, row) in enumerate(largest.iterrows()):
                print(f"{i+1}. 환자 {row['patient_id']}, 방문일 {row['visit_date']}: 약 {row['estimated_tokens']:.1f} 토큰 ({row['text_length']} 문자)")
        
        return df_lengths
    
    return None

### 군집화

#### 군집화 실행

In [7]:
def form_clusters_from_visit_embeddings(visit_embeddings, n_clusters=5, random_state=42):
    """방문 임베딩으로부터 군집 형성"""

    # 임베딩 벡터 추출
    embeddings = np.array([item['embedding'] for item in visit_embeddings])
    
    # K-means 군집화
    kmeans = KMeans(n_clusters=n_clusters, random_state=random_state, n_init=10)
    clusters = kmeans.fit_predict(embeddings)
    
    # 각 방문에 군집 할당
    for i, item in enumerate(visit_embeddings):
        item['cluster'] = int(clusters[i])
    
    # 군집 중심 반환
    cluster_centers = kmeans.cluster_centers_
    
    return visit_embeddings, cluster_centers

#### 환자 & 방문 레코드들 군집 분배

In [9]:
def calculate_cluster_probabilities(patient_embeddings, cluster_centers, temperature=0.1):
    """각 환자의 모든 군집에 대한 소속 확률 계산"""
    import numpy as np
    from scipy.special import softmax
    from scipy.spatial.distance import cosine
    
    # 결과 리스트
    patients_with_probs = []
    
    for patient in patient_embeddings:
        embedding = np.array(patient['embedding'])
        
        # 모든 군집 중심과의 코사인 유사도 계산
        similarities = []
        for center in cluster_centers:
            similarity = 1 - cosine(embedding, center)  # 코사인 거리를 유사도로 변환
            similarities.append(similarity)
        
        # 유사도 통계 확인
        similarity_mean = np.mean(similarities)
        similarity_std = np.std(similarities)
        similarity_range = np.max(similarities) - np.min(similarities)
        
        # 유사도 차별화가 너무 작으면 온도 조정
        if similarity_range < 0.05:  # 유사도 차이가 작을 때
            adjusted_temperature = 0.01  # 낮은 온도로 차이 강조
        else:
            adjusted_temperature = temperature
        
        # 유사도를 확률로 변환 (온도 조정된 소프트맥스)
        probabilities = softmax(np.array(similarities) / adjusted_temperature)
        
        # 결과 저장
        patient_result = patient.copy()
        patient_result['cluster_probabilities'] = {i: float(prob) for i, prob in enumerate(probabilities)}
        patient_result['similarity_stats'] = {
            'mean': float(similarity_mean),
            'std': float(similarity_std),
            'range': float(similarity_range)
        }
        patients_with_probs.append(patient_result)
    
    return patients_with_probs

### 데이터프레임에 군집 정보 추가 함수

In [9]:
def add_patient_clusters_to_dataframe(df, patients_with_probs):
    """데이터프레임에 환자별 군집 소속 확률 추가"""
    import pandas as pd
    
    # 환자 ID와 군집 확률을 매핑하는 딕셔너리 생성
    patient_cluster_map = {}
    
    for patient in patients_with_probs:
        patient_id = patient['patient_id']
        cluster_probs = patient['cluster_probabilities']
        
        # 가장 높은 확률을 가진 군집
        max_prob_cluster = max(cluster_probs.items(), key=lambda x: x[1])[0]
        
        patient_cluster_map[patient_id] = {
            'most_likely_cluster': max_prob_cluster,
            'cluster_probabilities': cluster_probs
        }
    
    # 결과 데이터프레임 생성
    df_result = df.copy()
    
    # 가장 높은 확률의 군집 컬럼 추가
    df_result['most_likely_cluster'] = df_result['환자번호'].map(
        lambda x: patient_cluster_map.get(x, {}).get('most_likely_cluster', -1)
    )
    
    # 각 군집별 확률 컬럼 추가
    n_clusters = len(next(iter(patient_cluster_map.values()))['cluster_probabilities']) if patient_cluster_map else 0
    
    for i in range(n_clusters):
        df_result[f'cluster_{i}_prob'] = df_result['환자번호'].map(
            lambda x: patient_cluster_map.get(x, {}).get('cluster_probabilities', {}).get(i, 0.0)
        )
    
    return df_result

### 임베딩-군집화 실행 파이프라인
- 환자 단위 군집화로 변경

In [7]:
def run_patient_level_clustering_pipeline(df, api_key, n_clusters=5, temperature=0.1, batch_size=20):
    """환자 단위 임베딩 및 군집화 파이프라인 실행 함수"""
    import os
    
    # 임베딩 파일 경로 설정
    embedding_dir = '/Users/nam-yeong/git/prj_centum/gpt_word/final_result/embeddings'
    patient_embedding_pattern = 'patient_embeddings_batch_*.json'
    
    # 기존 임베딩 파일 확인
    import glob
    existing_files = glob.glob(os.path.join(embedding_dir, patient_embedding_pattern))
    
    # 1. 환자 단위 임베딩 생성 또는 로드
    if existing_files:
        print(f"기존 환자 임베딩 파일 {len(existing_files)}개가 발견되었습니다.")
        print("임베딩 파일을 로드합니다...")
        patient_embeddings = load_patient_embeddings(existing_files)
    else:
        print("환자별 임베딩 생성 중...")
        patient_embeddings = create_patient_level_embeddings(df, api_key, batch_size=batch_size)
        
        # 통합 파일 저장
        save_embeddings_to_json(
            patient_embeddings, 
            f'patient_embeddings_complete_{len(patient_embeddings)}patients.json'
        )
    
    # 임베딩이 없는 경우 처리
    if not patient_embeddings:
        raise ValueError("유효한 임베딩을 생성하거나 로드할 수 없습니다.")
    
    # 2. 군집 형성
    print(f"{n_clusters}개 군집 형성 중...")
    patient_embeddings, cluster_centers = form_clusters_from_patient_embeddings(
        patient_embeddings, n_clusters=n_clusters
    )
    
    # 3. 각 환자의 군집 소속 확률 계산
    print("군집 소속 확률 계산 중...")
    patients_with_probs = calculate_cluster_probabilities(
        patient_embeddings, cluster_centers, temperature=temperature
    )
    
    # 4. 원본 데이터프레임에 확률 추가
    print("데이터프레임에 군집 확률 열 추가 중...")
    df_with_clusters = add_patient_clusters_to_dataframe(df, patients_with_probs)
    
    # 5. 군집 특성 분석
    print("군집 특성 분석 중...")
    cluster_features = analyze_patient_cluster_characteristics(df_with_clusters, n_clusters)
    
    print("환자별 군집화 및 확률 할당 완료!")
    return df_with_clusters, cluster_centers, cluster_features

### 환자 임베딩 기반 군집화 함수

In [8]:
def form_clusters_from_patient_embeddings(patient_embeddings, n_clusters=5, random_state=42):
    """환자 임베딩으로부터 군집 형성"""
    from sklearn.cluster import KMeans
    import numpy as np
    
    # 임베딩 벡터 추출
    embeddings = np.array([item['embedding'] for item in patient_embeddings])
    
    # K-means 군집화
    kmeans = KMeans(n_clusters=n_clusters, random_state=random_state, n_init=10)
    clusters = kmeans.fit_predict(embeddings)
    
    # 각 환자에 군집 할당
    for i, item in enumerate(patient_embeddings):
        item['cluster'] = int(clusters[i])
    
    # 군집 중심 반환
    cluster_centers = kmeans.cluster_centers_
    
    return patient_embeddings, cluster_centers

### 군집 특성 상세 분석 함수

In [11]:
def analyze_patient_cluster_characteristics(df_with_clusters, n_clusters=5):
    """각 군집의 특성 상세 분석"""
    import pandas as pd
    import numpy as np
    
    # 환자 단위로 집계
    patient_df = df_with_clusters.groupby('환자번호').agg({
        'most_likely_cluster': 'first',  # 가장 높은 확률의 군집
        **{f'cluster_{i}_prob': 'first' for i in range(n_clusters)},  # 각 군집별 확률
        'CC_vas': ['first', 'last', 'mean'],  # 통증 지표
        'CMO_before': ['first', 'last', 'mean'],  # 개구량 지표
        'MMO_before': ['first', 'last', 'mean'],  # 최대 개구량 지표
        'dif_MMO_CMO': ['first', 'last', 'mean'],  # MMO-CMO 차이
        'Noise_Code': lambda x: x.value_counts().index[0] if not x.empty else 'Unknown',  # 가장 빈번한 소리 유형
        'CC_pain_type': lambda x: x.value_counts().index[0] if not x.empty else 'Unknown',  # 가장 빈번한 통증 유형
        '습관_habit_type': lambda x: x.value_counts().index[0] if not x.empty else 'Unknown',  # 가장 빈번한 습관
        'CC_location': lambda x: x.value_counts().index[0] if not x.empty else 'Unknown'  # 가장 빈번한 위치
    }).reset_index()
    
    # 컬럼명 평탄화
    patient_df.columns = ['_'.join(col).strip('_') if isinstance(col, tuple) else col for col in patient_df.columns]
    
    # 각 군집의 특성 분석
    cluster_features = {}
    
    for cluster_id in range(n_clusters):
        # 현재 군집에 50% 이상 확률로 속한 환자
        cluster_patients = patient_df[patient_df[f'cluster_{cluster_id}_prob'] > 0.5]
        
        if len(cluster_patients) == 0:
            cluster_features[cluster_id] = {
                'size': 0,
                'description': "충분한 데이터 없음"
            }
            continue
        
        # 기본 통계
        size = len(cluster_patients)
        
        # 수치형 변수 통계
        numeric_stats = {}
        for metric in ['CC_vas', 'CMO_before', 'MMO_before', 'dif_MMO_CMO']:
            metric_first = f'{metric}_first'
            metric_last = f'{metric}_last'
            
            if metric_first in cluster_patients.columns and metric_last in cluster_patients.columns:
                vals_first = cluster_patients[metric_first].dropna()
                vals_last = cluster_patients[metric_last].dropna()
                
                if len(vals_first) > 0 and len(vals_last) > 0:
                    # 시작-종료 변화 계산
                    mean_first = vals_first.mean()
                    mean_last = vals_last.mean()
                    mean_change = mean_last - mean_first
                    mean_pct_change = (mean_change / mean_first * 100) if mean_first != 0 else float('inf')
                    
                    numeric_stats[metric] = {
                        'mean_first': float(mean_first),
                        'mean_last': float(mean_last),
                        'mean_change': float(mean_change),
                        'mean_pct_change': float(mean_pct_change)
                    }
        
        # 범주형 변수 분포
        categorical_stats = {}
        for cat in ['Noise_Code', 'CC_pain_type', '습관_habit_type', 'CC_location']:
            if cat in cluster_patients.columns:
                values = cluster_patients[cat].dropna()
                if len(values) > 0:
                    # 가장 빈번한 범주
                    value_counts = values.value_counts(normalize=True)
                    top_categories = value_counts.head(3).to_dict() if len(value_counts) >= 3 else value_counts.to_dict()
                    categorical_stats[cat] = top_categories
        
        # 회복률 계산
        if 'CC_vas_first' in cluster_patients.columns and 'CC_vas_last' in cluster_patients.columns:
            improved_count = sum((cluster_patients['CC_vas_last'] - cluster_patients['CC_vas_first']) < -1)  # 통증 1 이상 감소
            recovery_rate = improved_count / size if size > 0 else 0
        else:
            recovery_rate = None
        
        # 다른 군집과 비교 (t-검정)
        significant_diffs = []
        other_patients = patient_df[patient_df[f'cluster_{cluster_id}_prob'] <= 0.5]
        
        for metric in ['CC_vas_last', 'CMO_before_last', 'MMO_before_last']:
            if metric in cluster_patients.columns and metric in other_patients.columns:
                try:
                    from scipy.stats import ttest_ind
                    
                    cluster_vals = cluster_patients[metric].dropna()
                    other_vals = other_patients[metric].dropna()
                    
                    if len(cluster_vals) >= 5 and len(other_vals) >= 5:
                        t_stat, p_value = ttest_ind(cluster_vals, other_vals, equal_var=False)
                        
                        if p_value < 0.05:  # 통계적으로 유의미한 차이
                            effect_size = (cluster_vals.mean() - other_vals.mean()) / \
                                          np.sqrt((cluster_vals.std()**2 + other_vals.std()**2) / 2)
                            
                            significant_diffs.append({
                                'metric': metric,
                                'cluster_mean': float(cluster_vals.mean()),
                                'other_mean': float(other_vals.mean()),
                                'diff': float(cluster_vals.mean() - other_vals.mean()),
                                'p_value': float(p_value),
                                'effect_size': float(effect_size)
                            })
                except Exception as e:
                    print(f"t-검정 수행 중 오류: {str(e)}")
        
        # 군집 설명 생성
        description = generate_cluster_description(numeric_stats, categorical_stats, recovery_rate, significant_diffs)
        
        # 결과 저장
        cluster_features[cluster_id] = {
            'size': size,
            'numeric_stats': numeric_stats,
            'categorical_stats': categorical_stats,
            'recovery_rate': recovery_rate,
            'significant_diffs': significant_diffs,
            'description': description
        }
        
        # 결과 출력
        print(f"\n군집 {cluster_id} (환자 수: {size}명):")
        print(f"설명: {description}")
        print("주요 특성:")
        
        for metric, stats in numeric_stats.items():
            direction = "감소" if stats['mean_change'] < 0 else "증가"
            print(f"- {metric}: {stats['mean_first']:.2f} → {stats['mean_last']:.2f} ({abs(stats['mean_change']):.2f} {direction}, {abs(stats['mean_pct_change']):.1f}%)")
        
        print("주요 범주:")
        for cat, values in categorical_stats.items():
            top_cat = list(values.keys())[0] if values else "정보 없음"
            top_pct = list(values.values())[0] if values else 0
            print(f"- {cat}: {top_cat} ({top_pct:.1%})")
        
        if recovery_rate is not None:
            print(f"회복률: {recovery_rate:.1%}")
        
        if significant_diffs:
            print("다른 군집과의 유의미한 차이:")
            for diff in significant_diffs:
                print(f"- {diff['metric']}: {diff['cluster_mean']:.2f} vs {diff['other_mean']:.2f} (p={diff['p_value']:.4f})")
    
    return cluster_features

In [12]:
def generate_cluster_description(numeric_stats, categorical_stats, recovery_rate, significant_diffs):
    """군집 특성을 기반으로 설명 생성"""
    description_parts = []
    
    # 통증 관련 설명
    if 'CC_vas' in numeric_stats:
        vas_stats = numeric_stats['CC_vas']
        vas_first = vas_stats['mean_first']
        vas_change = vas_stats['mean_change']
        
        if vas_first > 7:
            pain_level = "심한 통증"
        elif vas_first > 4:
            pain_level = "중등도 통증"
        else:
            pain_level = "경미한 통증"
        
        if vas_change < -2:
            pain_change = "큰 개선"
        elif vas_change < -0.5:
            pain_change = "부분 개선"
        elif vas_change < 0.5:
            pain_change = "유지"
        else:
            pain_change = "악화"
        
        description_parts.append(f"{pain_level} ({pain_change})")
    
    # 개구량 관련 설명
    if 'CMO_before' in numeric_stats:
        cmo_stats = numeric_stats['CMO_before']
        cmo_first = cmo_stats['mean_first']
        cmo_change = cmo_stats['mean_change']
        
        if cmo_first < 30:
            opening_level = "개구량 제한"
        elif cmo_first < 40:
            opening_level = "부분 개구량"
        else:
            opening_level = "정상 개구량"
        
        if cmo_change > 5:
            opening_change = "큰 개선"
        elif cmo_change > 2:
            opening_change = "부분 개선"
        elif cmo_change > -2:
            opening_change = "유지"
        else:
            opening_change = "악화"
        
        description_parts.append(f"{opening_level} ({opening_change})")
    
    # 증상 관련 설명
    symptom_parts = []
    
    if 'CC_pain_type' in categorical_stats:
        top_pain = list(categorical_stats['CC_pain_type'].keys())[0] if categorical_stats['CC_pain_type'] else "정보 없음"
        if top_pain != "Unknown" and top_pain != "정보 없음":
            symptom_parts.append(top_pain)
    
    if 'CC_location' in categorical_stats:
        top_location = list(categorical_stats['CC_location'].keys())[0] if categorical_stats['CC_location'] else "정보 없음"
        if top_location != "Unknown" and top_location != "정보 없음":
            symptom_parts.append(f"{top_location}부위")
    
    if 'Noise_Code' in categorical_stats:
        top_noise = list(categorical_stats['Noise_Code'].keys())[0] if categorical_stats['Noise_Code'] else "정보 없음"
        if top_noise != "Unknown" and top_noise != "No-Noise" and top_noise != "정보 없음":
            symptom_parts.append(f"{top_noise} 관절음")
    
    if symptom_parts:
        description_parts.append(", ".join(symptom_parts))
    
    # 회복률 관련 설명
    if recovery_rate is not None:
        if recovery_rate > 0.7:
            description_parts.append("높은 회복률")
        elif recovery_rate > 0.4:
            description_parts.append("중간 회복률")
        elif recovery_rate > 0.2:
            description_parts.append("낮은 회복률")
    
    # 설명 조합
    if description_parts:
        return ", ".join(description_parts)
    else:
        return "특성 정보 부족"

### 군집 및 패턴 식별

#### 완치 환자 식별

In [13]:
def identify_recovered_patients(df, recovery_threshold=70):
    """기존 calculate_composite_recovery_score 함수를 활용한 완치 환자 식별 함수
    
    Parameters:
    -----------
    df : DataFrame
        환자 데이터가 포함된 데이터프레임
    recovery_threshold : float, optional (default=70)
        완치로 간주할 복합 점수 임계값
    
    Returns:
    --------
    list
        완치된 환자 ID 목록
    dict
        환자별 완치 점수 및 등급 정보
    """
    # 환자별 마지막 방문 데이터 추출
    patient_last_visits = df.sort_values('날짜').groupby('환자번호').last().reset_index()
    
    # 결과 저장할 컨테이너
    recovered_patients = []
    patient_details = {}
    recovery_grades = {
        "완전 회복 (Complete Recovery)": 0,
        "상당한 회복 (Substantial Recovery)": 0,
        "중간 회복 (Moderate Recovery)": 0,
        "경미한 회복 (Mild Recovery)": 0,
        "최소 회복 (Minimal Recovery)": 0,
        "회복 미미 (Little to No Recovery)": 0
    }
    
    # 각 환자에 대해 복합 점수 계산
    for _, row in patient_last_visits.iterrows():
        patient_id = row['환자번호']
        
        # 기존 calculate_composite_recovery_score 함수 활용
        composite_score, details = calculate_composite_recovery_score(row)
        
        # 상세 정보 저장
        patient_details[patient_id] = {
            'composite_score': composite_score,
            'recovery_grade': details['recovery_grade'],
            'score_breakdown': details['scores']
        }
        
        # 회복 등급 통계 업데이트
        recovery_grades[details['recovery_grade']] += 1
        
        # 임계값 이상인 환자는 회복된 것으로 간주
        if composite_score >= recovery_threshold:
            recovered_patients.append(patient_id)
    
    # 결과 출력
    print(f"복합 완치 점수 {recovery_threshold} 이상인 환자: {len(recovered_patients)}명")
    print(f"회복 등급 분포:")
    for grade, count in recovery_grades.items():
        print(f"  - {grade}: {count}명")
    
    return recovered_patients, patient_details

#### 완치 환자 군집 확률 궤적 분석

In [14]:
def analyze_recovered_patient_trajectories(df_with_probs, recovered_ids, n_clusters=5):
    """완치 환자들의 군집 확률 궤적 분석"""
    # 완치 환자 데이터 필터링
    recovered_df = df_with_probs[df_with_probs['환자번호'].isin(recovered_ids)]
    
    # 치료 단계별 확률 패턴 분석
    stage_patterns = {}
    
    # 각 환자별로 치료 과정을 정규화하여 분석
    for patient_id, patient_visits in recovered_df.groupby('환자번호'):
        # 날짜순 정렬
        patient_visits = patient_visits.sort_values('날짜')
        
        # 치료 기간 계산
        treatment_days = (patient_visits['날짜'].max() - patient_visits['날짜'].min()).days
        if treatment_days == 0:  # 방문이 하루만 있는 경우
            continue
            
        # 방문별 상대적 진행 단계 계산 (0~100%)
        start_date = patient_visits['날짜'].min()
        
        for _, visit in patient_visits.iterrows():
            progress = ((visit['날짜'] - start_date).days / treatment_days * 100)
            stage = min(int(progress // 10) * 10, 90)  # 10% 단위로 구분 (0%, 10%, ..., 90%)
            
            if stage not in stage_patterns:
                stage_patterns[stage] = {f'cluster_{i}_prob': [] for i in range(n_clusters)}
            
            for i in range(n_clusters):
                prob_col = f'cluster_{i}_prob'
                if prob_col in visit:
                    stage_patterns[stage][prob_col].append(visit[prob_col])
    
    # 단계별 평균 확률 계산
    avg_stage_patterns = {}
    for stage, probs in stage_patterns.items():
        avg_stage_patterns[stage] = {
            cluster: np.mean(values) if values else 0 
            for cluster, values in probs.items()
        }
    
    # 데이터 시각화
    visualize_recovery_trajectories(avg_stage_patterns, n_clusters)
    
    return avg_stage_patterns

In [15]:
def visualize_recovery_trajectories(avg_stage_patterns, n_clusters):
    """완치 환자들의 치료 단계별 군집 확률 시각화"""
    import matplotlib.pyplot as plt
    
    # 데이터 준비
    stages = sorted(avg_stage_patterns.keys())
    cluster_probs = {f'cluster_{i}_prob': [] for i in range(n_clusters)}
    
    for stage in stages:
        for i in range(n_clusters):
            prob_col = f'cluster_{i}_prob'
            cluster_probs[prob_col].append(avg_stage_patterns[stage][prob_col])
    
    # 시각화
    plt.figure(figsize=(12, 6))
    for i in range(n_clusters):
        prob_col = f'cluster_{i}_prob'
        plt.plot(stages, cluster_probs[prob_col], marker='o', linewidth=2, label=f'군집 {i}')
    
    plt.title('완치 환자의 치료 단계별 군집 소속 확률 변화', fontsize=14)
    plt.xlabel('치료 진행 단계 (%)', fontsize=12)
    plt.ylabel('평균 군집 소속 확률', fontsize=12)
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.legend()
    plt.xticks(stages)
    plt.tight_layout()
    plt.show()
    
    return plt

### 완치 궤적을 기반으로한 회복 예측 모델 구축

#### 많은 데이터 case : 로지스틱 회귀

In [16]:
def build_recovery_prediction_model(df_with_probs, recovered_ids, clinical_metrics=None):
    """완치 궤적 기반 회복 예측 모델 구축"""
    # 임상 지표가 제공되지 않은 경우 기본값 설정
    if clinical_metrics is None:
        clinical_metrics = ['CC_vas', 'CMO_before', 'MMO_before', 'dif_MMO_CMO']
    
    # 완치/비완치 환자 분류
    df_with_probs['is_recovered'] = df_with_probs['환자번호'].isin(recovered_ids)
    
    # 사용할 특성 선택
    cluster_cols = [col for col in df_with_probs.columns if col.startswith('cluster_') and col.endswith('_prob')]
    metrics_to_use = [metric for metric in clinical_metrics if metric in df_with_probs.columns]
    if not metrics_to_use:
        print("경고: 제공된 임상 지표가 데이터프레임에 없습니다. 군집 확률만 사용합니다.")
    
    feature_cols = cluster_cols + metrics_to_use
    
    # 환자 횟수가 적을 경우 로지스틱 회귀 대신 간단한 가중 모델 사용
    if len(df_with_probs['환자번호'].unique()) < 30:
        print("환자 수가 적어 통계적 회귀 모델 대신 가중치 기반 모델을 사용합니다.")
        return build_simple_weighted_model(df_with_probs, recovered_ids, feature_cols)
    
    # 충분한 데이터가 있으면 로지스틱 회귀 수행
    from sklearn.linear_model import LogisticRegression
    from sklearn.model_selection import train_test_split, cross_val_score
    from sklearn.metrics import classification_report, roc_auc_score
    
    # 특성 및 타겟 데이터 준비
    X = df_with_probs[feature_cols].fillna(0)
    y = df_with_probs['is_recovered']
    
    # 환자별로 데이터 분할을 위한 그룹 정보
    groups = df_with_probs['환자번호'].values
    
    # 모델 초기화
    model = LogisticRegression(random_state=42, class_weight='balanced', max_iter=1000)
    
    # 교차 검증 (환자별 그룹화 고려)
    try:
        from sklearn.model_selection import GroupKFold
        gkf = GroupKFold(n_splits=5)
        cv_scores = cross_val_score(model, X, y, cv=gkf.split(X, y, groups), scoring='roc_auc')
        print(f"교차 검증 ROC-AUC: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")
    except Exception as e:
        print(f"환자별 교차 검증 실패: {e}")
        # 일반 교차 검증으로 대체
        cv_scores = cross_val_score(model, X, y, cv=5, scoring='roc_auc')
        print(f"일반 교차 검증 ROC-AUC: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")
    
    # 전체 데이터로 모델 학습
    model.fit(X, y)
    
    # 특성 중요도 계산
    feature_importance = {
        feature: importance for feature, importance in zip(feature_cols, model.coef_[0])
    }
    
    # 중요도 시각화
    visualize_feature_importance(feature_importance)
    
    return model, feature_importance

#### 적은 데이터 case : 가중 모델

In [17]:
def build_simple_weighted_model(df_with_probs, recovered_ids, feature_cols):
    """적은 데이터에 적합한 간단한 가중 모델 구축"""
    # 완치 환자와 비완치 환자 구분
    recovered_df = df_with_probs[df_with_probs['환자번호'].isin(recovered_ids)]
    non_recovered_df = df_with_probs[~df_with_probs['환자번호'].isin(recovered_ids)]
    
    # 각 특성의 평균값 계산
    recovered_means = recovered_df[feature_cols].mean()
    non_recovered_means = non_recovered_df[feature_cols].mean()
    
    # 특성별 차이 계산
    feature_diffs = recovered_means - non_recovered_means
    
    # 효과 크기 계산 (Standardized Mean Difference)
    recovered_std = recovered_df[feature_cols].std()
    non_recovered_std = non_recovered_df[feature_cols].std()
    
    # 0으로 나누기 방지
    pooled_std = np.sqrt((recovered_std**2 + non_recovered_std**2) / 2)
    pooled_std = pooled_std.replace(0, 1)  # 0인 경우 1로 대체
    
    effect_sizes = feature_diffs / pooled_std
    
    # 효과 크기로 가중치 정규화
    total_effect = np.abs(effect_sizes).sum()
    if total_effect == 0:
        weights = pd.Series(1/len(effect_sizes), index=effect_sizes.index)
    else:
        weights = np.abs(effect_sizes) / total_effect
    
    # 모델 함수 정의 (가중 평균 접근법)
    def predict_proba(X):
        # 정규화된 특성 생성
        X_norm = X.copy()
        for col in X.columns:
            mean_val = (recovered_means[col] + non_recovered_means[col]) / 2
            std_val = pooled_std[col]
            X_norm[col] = (X[col] - mean_val) / std_val
        
        # 가중 점수 계산
        scores = np.zeros(len(X))
        for col in X.columns:
            direction = np.sign(effect_sizes[col])
            scores += direction * X_norm[col] * weights[col]
        
        # 확률로 변환
        from scipy.special import expit  # 시그모이드 함수
        probs = expit(scores)
        
        # [not_recovered_prob, recovered_prob] 형태로 반환
        return np.column_stack([1 - probs, probs])
    
    # 간단한 모델 객체 생성
    class SimpleWeightedModel:
        def __init__(self, weights, effect_sizes, recovered_means, non_recovered_means, pooled_std, feature_names):
            self.weights = weights
            self.effect_sizes = effect_sizes
            self.recovered_means = recovered_means
            self.non_recovered_means = non_recovered_means
            self.pooled_std = pooled_std
            self.feature_names_in_ = feature_names
        
        def predict_proba(self, X):
            return predict_proba(X)
        
        def predict(self, X):
            probs = self.predict_proba(X)
            return (probs[:, 1] >= 0.5).astype(int)
    
    model = SimpleWeightedModel(
        weights, effect_sizes, recovered_means, non_recovered_means, 
        pooled_std, feature_cols
    )
    
    # 특성 중요도 (가중치로 대체)
    feature_importance = {
        feature: weight for feature, weight in zip(feature_cols, weights)
    }
    
    # 중요도 시각화
    visualize_feature_importance(feature_importance)
    
    return model, feature_importance

#### 특성 중요도 시각화

In [14]:
def visualize_patient_embeddings(patient_embeddings, cluster_centers=None, n_clusters=5):
    """환자 임베딩 2D 시각화"""
    import matplotlib.pyplot as plt
    import numpy as np
    from sklearn.manifold import TSNE
    
    # 임베딩 벡터 추출
    embeddings = np.array([item['embedding'] for item in patient_embeddings])
    
    # t-SNE 차원 축소
    tsne = TSNE(n_components=2, random_state=42, perplexity=min(30, max(5, len(embeddings)//5)))
    embeddings_2d = tsne.fit_transform(embeddings)
    
    # 군집별 색상 지정
    colors = plt.cm.rainbow(np.linspace(0, 1, n_clusters))
    
    # 군집별 시각화
    plt.figure(figsize=(12, 10))
    
    # 군집 정보가 있으면 군집별로 플롯
    if 'cluster' in patient_embeddings[0]:
        for i in range(n_clusters):
            # 현재 군집에 속한 환자 인덱스
            idx = [j for j, item in enumerate(patient_embeddings) if item['cluster'] == i]
            
            if idx:
                plt.scatter(
                    embeddings_2d[idx, 0], 
                    embeddings_2d[idx, 1],
                    color=colors[i],
                    label=f'군집 {i} ({len(idx)}명)',
                    alpha=0.7
                )
        
        # 군집 중심 시각화 (있는 경우)
        if cluster_centers is not None:
            # 군집 중심도 t-SNE 변환 (같은 변환 적용)
            centers_2d = tsne.fit_transform(np.vstack([embeddings, cluster_centers]))[-n_clusters:]
            
            plt.scatter(
                centers_2d[:, 0],
                centers_2d[:, 1],
                s=200,
                marker='X',
                color='black',
                label='군집 중심'
            )
    else:
        # 군집 정보가 없으면 단일 색상으로 표시
        plt.scatter(
            embeddings_2d[:, 0], 
            embeddings_2d[:, 1],
            alpha=0.7
        )
    
    plt.title('환자 임베딩 t-SNE 시각화', fontsize=15)
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()
    
    return embeddings_2d

In [18]:
def visualize_feature_importance(feature_importance):
    """특성 중요도 시각화"""
    import matplotlib.pyplot as plt
    
    # 중요도 정렬
    sorted_importance = sorted(feature_importance.items(), key=lambda x: abs(x[1]), reverse=True)
    features = [x[0] for x in sorted_importance]
    importance = [x[1] for x in sorted_importance]
    
    # 너무 많은 특성이 있으면 상위 10개만 표시
    if len(features) > 10:
        features = features[:10]
        importance = importance[:10]
    
    # 시각화
    plt.figure(figsize=(10, 6))
    colors = ['g' if imp > 0 else 'r' for imp in importance]
    bars = plt.barh(range(len(features)), [abs(imp) for imp in importance], color=colors)
    
    plt.yticks(range(len(features)), features)
    plt.xlabel('특성 중요도 (절대값)', fontsize=12)
    plt.title('완치 예측에 중요한 특성', fontsize=14)
    
    # 양수/음수 구분을 위한 범례
    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor='g', label='완치와 양의 상관관계'),
        Patch(facecolor='r', label='완치와 음의 상관관계')
    ]
    plt.legend(handles=legend_elements)
    
    plt.tight_layout()
    plt.show()
    
    # 중요도 출력
    print("특성 중요도 (상위):")
    for feature, importance in sorted_importance[:10]:
        sign = "+" if importance > 0 else "-"
        print(f"  {sign} {feature}: {abs(importance):.4f}")
    
    return plt

### 신규 환자 적용

#### 신규 환자 훼복 궤적 예측

In [19]:
def predict_recovery_trajectory(patient_data, df_with_probs, model, avg_stage_patterns, cluster_centers):
    """신규 환자의 회복 궤적 예측"""
    import numpy as np
    from scipy.spatial.distance import cosine
    from scipy.special import softmax
    
    # 환자 ID 확인
    patient_id = patient_data['환자번호']
    
    # 환자 방문 데이터 추출
    patient_visits = df_with_probs[df_with_probs['환자번호'] == patient_id].sort_values('날짜')
    
    if len(patient_visits) == 0:
        raise ValueError(f"환자 ID {patient_id}에 대한 데이터를 찾을 수 없습니다.")
    
    # 가장 최근 방문 사용
    latest_visit = patient_visits.iloc[-1]
    
    # 군집 확률 추출
    n_clusters = len(cluster_centers)
    cluster_probs = {
        f'cluster_{i}_prob': latest_visit[f'cluster_{i}_prob'] 
        for i in range(n_clusters) if f'cluster_{i}_prob' in latest_visit
    }
    
    # 특성 및 타겟 데이터 준비
    feature_cols = list(model.feature_names_in_)
    
    # 특성 벡터 구성
    features = []
    for col in feature_cols:
        if col in latest_visit:
            features.append(latest_visit[col])
        else:
            features.append(0)  # 누락된 특성은 0으로 대체
    
    features_array = np.array([features])
    
    # 회복 가능성 예측
    recovery_prob = model.predict_proba(features_array)[0][1]
    
    # 현재 군집 확률 분포 추출
    current_probs = np.array([[latest_visit[f'cluster_{i}_prob'] for i in range(n_clusters)]])
    
    # 가장 유사한 진행 단계 찾기
    min_distance = float('inf')
    best_stage = None
    
    for stage, avg_probs in avg_stage_patterns.items():
        stage_probs = np.array([[avg_probs.get(f'cluster_{i}_prob', 0) for i in range(n_clusters)]])
        
        # 유클리드 거리 계산
        distance = np.linalg.norm(current_probs - stage_probs)
        
        if distance < min_distance:
            min_distance = distance
            best_stage = stage
    
    # 주요 임상 지표 현재 값
    clinical_metrics = {}
    for metric in ['CC_vas', 'CMO_before', 'MMO_before', 'dif_MMO_CMO']:
        if metric in latest_visit:
            clinical_metrics[metric] = float(latest_visit[metric])
    
    # 완치 군집 추세 대비 현재 위치 시각화
    visualize_patient_vs_recovery_trend(current_probs[0], avg_stage_patterns, n_clusters, best_stage)
    
    # 결과 반환
    prediction_result = {
        'patient_id': patient_id,
        'recovery_probability': float(recovery_prob),
        'current_stage': best_stage,
        'cluster_probabilities': cluster_probs,
        'most_likely_cluster': int(np.argmax([cluster_probs.get(f'cluster_{i}_prob', 0) for i in range(n_clusters)])),
        'estimated_remaining_stages': 10 - (best_stage // 10) if best_stage is not None else None,
        'current_clinical_metrics': clinical_metrics
    }
    
    return prediction_result

In [20]:
def visualize_patient_vs_recovery_trend(current_probs, avg_stage_patterns, n_clusters, current_stage):
    """환자의 현재 군집 확률과 완치 환자의 트렌드 비교 시각화"""
    import matplotlib.pyplot as plt
    
    # 데이터 준비
    stages = sorted(avg_stage_patterns.keys())
    recovery_probs = {}
    
    for i in range(n_clusters):
        col = f'cluster_{i}_prob'
        recovery_probs[col] = [avg_stage_patterns[stage].get(col, 0) for stage in stages]
    
    # 시각화
    plt.figure(figsize=(12, 6))
    
    # 전체 트렌드 그리기
    for i in range(n_clusters):
        col = f'cluster_{i}_prob'
        plt.plot(stages, recovery_probs[col], '--', alpha=0.7, linewidth=1, label=f'군집 {i} 트렌드')
    
    # 현재 위치 강조
    current_x = current_stage if current_stage is not None else 0
    for i in range(n_clusters):
        col = f'cluster_{i}_prob'
        plt.scatter([current_x], [current_probs[i]], s=100, label=f'현재 군집 {i}')
    
    # 예상 경로 표시
    if current_stage is not None:
        future_stages = [s for s in stages if s > current_stage]
        for i in range(n_clusters):
            col = f'cluster_{i}_prob'
            future_probs = [avg_stage_patterns[stage].get(col, 0) for stage in future_stages]
            if future_stages and future_probs:
                plt.plot(future_stages, future_probs, 'g-', alpha=0.8, linewidth=2)
        
        # 현재 위치 강조
        plt.axvline(x=current_stage, color='r', linestyle=':', alpha=0.6, label='현재 단계')
    
    plt.title('환자의 현재 군집 확률 vs 완치 환자 트렌드', fontsize=14)
    plt.xlabel('치료 진행 단계 (%)', fontsize=12)
    plt.ylabel('군집 소속 확률', fontsize=12)
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
    plt.xticks(stages)
    plt.tight_layout()
    plt.show()
    
    return plt

### 성능 평가

In [21]:
def evaluate_model_statistics(df_with_probs, model, recovered_ids):
    """모델의 통계적 성능 평가"""
    from sklearn.metrics import roc_curve, auc, confusion_matrix, classification_report
    import matplotlib.pyplot as plt
    import numpy as np
    
    # 실제 완치 상태
    df_with_probs['is_recovered'] = df_with_probs['환자번호'].isin(recovered_ids)
    
    # 특성 컬럼
    feature_cols = list(model.feature_names_in_)
    
    # 예측 확률
    X = df_with_probs[feature_cols].fillna(0)
    y_true = df_with_probs['is_recovered']
    y_pred_proba = model.predict_proba(X)[:, 1]
    y_pred = (y_pred_proba >= 0.5).astype(int)
    
    # ROC 곡선
    fpr, tpr, thresholds = roc_curve(y_true, y_pred_proba)
    roc_auc = auc(fpr, tpr)
    
    plt.figure(figsize=(10, 8))
    plt.subplot(2, 1, 1)
    plt.plot(fpr, tpr, label=f'ROC 곡선 (AUC = {roc_auc:.3f})')
    plt.plot([0, 1], [0, 1], 'k--')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('완치 예측 모델 ROC 곡선')
    plt.legend(loc='lower right')
    
    # 혼동 행렬
    cm = confusion_matrix(y_true, y_pred)
    plt.subplot(2, 1, 2)
    plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
    plt.title('혼동 행렬')
    plt.colorbar()
    
    classes = ['비완치', '완치']
    tick_marks = np.arange(len(classes))
    plt.xticks(tick_marks, classes)
    plt.yticks(tick_marks, classes)
    
    thresh = cm.max() / 2.
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, format(cm[i, j], 'd'),
                     horizontalalignment="center",
                     color="white" if cm[i, j] > thresh else "black")
    
    plt.ylabel('실제 레이블')
    plt.xlabel('예측 레이블')
    plt.tight_layout()
    plt.show()
    
    # 분류 보고서
    report = classification_report(y_true, y_pred, target_names=classes)
    print("분류 보고서:")
    print(report)
    
    # 통계적 유의성 검정
    from scipy import stats
    
    # 완치 환자와 비완치 환자의 예측 확률 비교
    recovered_probs = y_pred_proba[y_true]
    non_recovered_probs = y_pred_proba[~y_true]
    
    # t-검정
    t_stat, p_value = stats.ttest_ind(recovered_probs, non_recovered_probs)
    print(f"\n완치/비완치 환자 예측 확률 t-검정:")
    print(f"t-통계량: {t_stat:.4f}, p-값: {p_value:.4f}")
    print(f"통계적 유의성: {'있음 (p < 0.05)' if p_value < 0.05 else '없음 (p >= 0.05)'}")
    
    return {
        'roc_auc': roc_auc,
        'confusion_matrix': cm,
        'classification_report': report,
        't_test': {'t_stat': t_stat, 'p_value': p_value}
    }

### 실행 파이프라인

In [15]:
def run_patient_clustering_pipeline(df, api_key, n_clusters=5, batch_size=20):
    """환자 단위 임베딩 및 군집화 전체 파이프라인 실행"""
    print("===== 환자 단위 임베딩 및 군집화 파이프라인 시작 =====")
    
    # 1. 환자 단위 임베딩 생성
    df_with_clusters, cluster_centers, cluster_features = run_patient_level_clustering_pipeline(
        df, api_key, n_clusters=n_clusters, batch_size=batch_size
    )
    
    # 2. 임베딩 시각화
    print("\n군집 시각화 중...")
    
    # 환자별 임베딩 데이터 구조화
    patient_embeddings = []
    patient_ids = df_with_clusters['환자번호'].unique()
    
    for patient_id in patient_ids:
        patient_data = df_with_clusters[df_with_clusters['환자번호'] == patient_id].iloc[0]
        cluster_probs = {i: patient_data[f'cluster_{i}_prob'] for i in range(n_clusters) if f'cluster_{i}_prob' in patient_data}
        
        most_likely_cluster = max(cluster_probs.items(), key=lambda x: x[1])[0] if cluster_probs else -1
        
        # 임베딩 데이터 (실제 임베딩 벡터는 없지만 시각화를 위해 구조 생성)
        patient_embeddings.append({
            'patient_id': patient_id,
            'cluster': most_likely_cluster,
            'cluster_probabilities': cluster_probs
        })
    
    # 시각화용 임베딩 데이터가 있으면 시각화
    if 'embedding' in patient_embeddings[0]:
        visualize_patient_embeddings(patient_embeddings, cluster_centers, n_clusters)
    
    # 3. 결과 통계 출력
    print("\n군집 분포:")
    cluster_counts = df_with_clusters['most_likely_cluster'].value_counts().sort_index()
    for cluster_id, count in cluster_counts.items():
        description = cluster_features.get(cluster_id, {}).get('description', "정보 없음")
        print(f"군집 {cluster_id}: {count}명 - {description}")
    
    # 4. 군집별 환자 예시 출력
    print("\n각 군집 대표 환자:")
    for cluster_id in range(n_clusters):
        # 해당 군집에 높은 확률로 소속된 환자
        cluster_patients = df_with_clusters[df_with_clusters[f'cluster_{cluster_id}_prob'] > 0.7]
        
        if not cluster_patients.empty:
            # 소속 확률이 가장 높은 환자
            top_patient = cluster_patients.loc[cluster_patients[f'cluster_{cluster_id}_prob'].idxmax()]
            patient_id = top_patient['환자번호']
            prob = top_patient[f'cluster_{cluster_id}_prob']
            
            print(f"군집 {cluster_id} 대표 환자: {patient_id} (소속 확률: {prob:.2f})")
    
    print("\n===== 환자 단위 임베딩 및 군집화 파이프라인 완료 =====")
    return df_with_clusters, cluster_centers, cluster_features

In [22]:
def run_complete_predictive_pipeline(df, api_key, n_clusters=5, temperature=0.1, recovery_threshold=70, batch_size=20, log_errors=True):
    """전체 회복 궤적 예측 파이프라인 실행 함수
    
    Parameters:
    -----------
    df : DataFrame
        환자 데이터가 포함된 데이터프레임
    api_key : str
        OpenAI API 키
    n_clusters : int, optional (default=5)
        생성할 군집 수
    temperature : float, optional (default=0.1)
        확률 계산 시 사용할 온도 매개변수 (낮을수록 더 극단적 확률 분포)
    recovery_threshold : float, optional (default=70)
        완치로 간주할 복합 점수 임계값
        
    Returns:
    --------
    dict
        모든 모델 구성요소 및 예측 함수를 포함하는 결과 사전
    """
    print("===== 턱관절 장애 환자 회복 궤적 예측 시스템 구축 =====")
    print(f"군집 수: {n_clusters}, 온도 매개변수: {temperature}, 완치 임계값: {recovery_threshold}")
    
    # 1. 방문별 임베딩 및 군집화
    print("\n[단계 1/5] 방문별 임베딩 생성 및 군집화...")
    df_with_probs, cluster_centers, cluster_features = run_visit_level_clustering_pipeline(
        df, api_key, n_clusters=n_clusters, temperature=temperature, batch_size=batch_size
    )
    
    # 2. 완치 환자 식별
    print("\n[단계 2/5] 완치 환자 식별...")
    recovered_ids, patient_details = identify_recovered_patients(df, recovery_threshold)
    
    # 3. 완치 환자 궤적 분석
    print("\n[단계 3/5] 완치 환자 궤적 분석...")
    if len(recovered_ids) > 0:
        avg_stage_patterns = analyze_recovered_patient_trajectories(
            df_with_probs, recovered_ids, n_clusters
        )
    else:
        print("경고: 식별된 완치 환자가 없습니다. 샘플 궤적을 사용합니다.")
        # 샘플 궤적 생성 (실제 데이터 없는 경우)
        avg_stage_patterns = create_sample_trajectories(n_clusters)
    
    # 4. 예측 모델 구축
    print("\n[단계 4/5] 회복 예측 모델 구축...")
    model, feature_importance = build_recovery_prediction_model(
        df_with_probs, recovered_ids, ['CC_vas', 'CMO_before', 'MMO_before', 'dif_MMO_CMO']
    )
    
    # 5. 모델 평가 및 통계적 검증
    print("\n[단계 5/5] 모델 통계적 검증...")
    model_stats = evaluate_model_statistics(df_with_probs, model, recovered_ids)
    
    # 6. 예측 시스템 구축 완료
    print("\n===== 예측 시스템 구축 완료 =====")
    
    # 예측 함수 래핑
    def predict_recovery(patient_id):
        patient_data = df[df['환자번호'] == patient_id].iloc[-1]
        return predict_recovery_trajectory(
            patient_data, df_with_probs, model, avg_stage_patterns, cluster_centers
        )
    
    # 결과 반환
    result = {
        'df_with_probs': df_with_probs,
        'cluster_centers': cluster_centers,
        'cluster_features': cluster_features,
        'recovered_ids': recovered_ids,
        'patient_details': patient_details,
        'avg_stage_patterns': avg_stage_patterns,
        'model': model,
        'feature_importance': feature_importance,
        'model_stats': model_stats,
        'predict_recovery': predict_recovery
    }
    
    return result

In [23]:
def run_example_prediction(prediction_system, patient_id):
    """예제 환자에 대한 예측 실행 및 결과 출력"""
    print(f"\n===== 환자 {patient_id}의 회복 궤적 예측 =====")
    
    try:
        # 예측 수행
        result = prediction_system['predict_recovery'](patient_id)
        
        # 결과 출력
        print(f"환자 ID: {result['patient_id']}")
        print(f"완치 확률: {result['recovery_probability']:.2%}")
        print(f"현재 단계: {result['current_stage']}% (완치 기준)")
        
        if 'estimated_remaining_stages' in result and result['estimated_remaining_stages'] is not None:
            print(f"예상 남은 단계: {result['estimated_remaining_stages']} (각 단계는 약 10%의 진행 의미)")
        
        print("\n현재 군집 소속 확률:")
        for cluster, prob in result['cluster_probabilities'].items():
            print(f"  - {cluster}: {prob:.2%}")
        
        print(f"\n가장 유사한 군집: {result['most_likely_cluster']}")
        
        print("\n현재 임상 지표:")
        for metric, value in result['current_clinical_metrics'].items():
            print(f"  - {metric}: {value}")
        
        # 추가 통계 정보가 있으면 출력
        if 'statistical_validity' in result:
            print("\n통계적 유의성:")
            print(f"  - p값: {result['statistical_validity']['p_value']:.4f}")
            print(f"  - 유의성: {'있음 (p<0.05)' if result['statistical_validity']['p_value'] < 0.05 else '없음 (p≥0.05)'}")
    
    except Exception as e:
        print(f"예측 오류: {e}")
        import traceback
        traceback.print_exc()

In [24]:
def create_interactive_dashboard(prediction_system, df):
    """예측 시스템의 결과를 보여주는 대화형 대시보드 생성 (Optional)"""
    try:
        import matplotlib.pyplot as plt
        from ipywidgets import interact, widgets
        import ipywidgets as widgets
        
        # 환자 목록 준비
        patient_ids = sorted(df['환자번호'].unique())
        
        # 환자 선택 위젯
        patient_dropdown = widgets.Dropdown(
            options=patient_ids,
            description='환자 선택:',
            style={'description_width': 'initial'}
        )
        
        # 예측 실행 함수
        def on_patient_change(patient_id):
            if patient_id:
                plt.close('all')  # 이전 그래프 닫기
                run_example_prediction(prediction_system, patient_id)
        
        # 위젯 연결
        interact(on_patient_change, patient_id=patient_dropdown)
        
        print("대화형 대시보드가 준비되었습니다. 환자를 선택하여 예측 결과를 확인하세요.")
    
    except ImportError:
        print("대화형 대시보드를 위해 ipywidgets가 필요합니다.")
        print("pip install ipywidgets를 실행하여 설치할 수 있습니다.")
        
        # 대체 기능: 몇 가지 예제 환자 예측
        sample_patients = df['환자번호'].unique()[:3]  # 처음 3명만
        for patient_id in sample_patients:
            run_example_prediction(prediction_system, patient_id)

### 샘플 실험

In [56]:
df.Tongue_ridging_Intensity.value_counts()

Tongue_ridging_Intensity
1    14426
0    11698
2     1671
3      105
Name: count, dtype: int64

In [37]:
df.환자번호.nunique()

df_sp = df.groupby('환자번호').size().reset_index(name='방문수')
df_sp['방문수_범주'] = df_sp['방문수'].apply(lambda x: '낮음' if x <= 3 else ('중간' if x <= 10 else '높음'))
df_sp.groupby('방문수_범주').size()
# 방문수_범주 분포를 반영한 환자번호 샘플링 (계층적 샘플링)

# 각 범주별 환자 수 확인
category_counts = df_sp.groupby('방문수_범주').size()

# 계층적 샘플링(stratified sampling) 수행
from sklearn.model_selection import StratifiedShuffleSplit

# 전체 샘플 수 설정
total_samples = 600  # 총 샘플링할 환자 수

# 계층적 샘플링 수행
stratified_split = StratifiedShuffleSplit(n_splits=1, test_size=total_samples/len(df_sp), random_state=42)
for _, sample_idx in stratified_split.split(df_sp, df_sp['방문수_범주']):
    stratified_sample = df_sp.iloc[sample_idx]

stratified_sample.환자번호.to_list()
df_sp = df[df['환자번호'].isin(stratified_sample.환자번호.to_list())]


In [27]:
# 파이프라인 실행
df_with_probs, cluster_centers, cluster_features = run_visit_level_clustering_pipeline(
    df_sp, api_key, n_clusters=5, temperature=0.1, batch_size=100
)

임베딩 생성 전 텍스트 길이 샘플 확인 중...
샘플 5개 방문 데이터에 대한 텍스트 길이 확인 중...


5it [00:00, 879.53it/s]


텍스트 길이 분석 결과:
- 평균 텍스트 길이: 774.6 문자
- 최대 텍스트 길이: 825 문자
- 최소 텍스트 길이: 722 문자
- 평균 단어 수: 171.4 단어
- 추정 평균 토큰 수: 228.5 토큰
기존 방문 임베딩 파일 32개가 발견되었습니다.
임베딩 파일을 로드합니다...


/Users/nam-yeong/git/prj_centum/gpt_word/final_result/embeddings/visit_embeddings_batch_16.json에서 100개 임베딩 로드 완료
/Users/nam-yeong/git/prj_centum/gpt_word/final_result/embeddings/visit_embeddings_batch_20.json에서 100개 임베딩 로드 완료
/Users/nam-yeong/git/prj_centum/gpt_word/final_result/embeddings/visit_embeddings_batch_21.json에서 100개 임베딩 로드 완료
/Users/nam-yeong/git/prj_centum/gpt_word/final_result/embeddings/visit_embeddings_batch_17.json에서 100개 임베딩 로드 완료
/Users/nam-yeong/git/prj_centum/gpt_word/final_result/embeddings/visit_embeddings_batch_1.json에서 100개 임베딩 로드 완료
/Users/nam-yeong/git/prj_centum/gpt_word/final_result/embeddings/visit_embeddings_batch_30.json에서 100개 임베딩 로드 완료
/Users/nam-yeong/git/prj_centum/gpt_word/final_result/embeddings/visit_embeddings_batch_26.json에서 100개 임베딩 로드 완료
/Users/nam-yeong/git/prj_centum/gpt_word/final_result/embeddings/visit_embeddings_batch_6.json에서 100개 임베딩 로드 완료
/Users/nam-yeong/git/prj_centum/gpt_word/final_result/embeddings/visit_embeddings_batch_10.json에서 

/var/folders/7z/944bgjcs659fsp58nc954qgm0000gn/T/ipykernel_84009/1448760255.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[f'cluster_{i}_prob'] = 0.0
/var/folders/7z/944bgjcs659fsp58nc954qgm0000gn/T/ipykernel_84009/1448760255.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[f'cluster_{i}_prob'] = 0.0
/var/folders/7z/944bgjcs659fsp58nc954qgm0000gn/T/ipykernel_84009/1448760255.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_ind

군집 특성 분석 중...
방문별 군집화 및 확률 할당 완료!


In [28]:
analyze_cluster_characteristics(df_with_probs, 5)

{0: {'size': 0, 'description': '충분한 데이터 없음'},
 1: {'size': 0, 'description': '충분한 데이터 없음'},
 2: {'size': 0, 'description': '충분한 데이터 없음'},
 3: {'size': 0, 'description': '충분한 데이터 없음'},
 4: {'size': 0, 'description': '충분한 데이터 없음'}}

In [44]:
cluster_centers

array([[-0.00555143, -0.00115343,  0.02574948, ...,  0.00354592,
         0.01673026, -0.00518386],
       [-0.00450177, -0.0005724 ,  0.02276497, ...,  0.00172894,
         0.0151695 , -0.00239115],
       [-0.00950409, -0.00056038,  0.02391509, ...,  0.00245279,
         0.01563521, -0.00096785],
       [-0.00362584, -0.00375471,  0.02034339, ...,  0.00339591,
         0.01529752, -0.00022416],
       [-0.00587422, -0.00134825,  0.02540528, ...,  0.00237079,
         0.01676647, -0.00174055]])

In [46]:
sp = df_with_probs.환자번호.sample().values[0]
df_with_probs[df_with_probs['환자번호'] == sp]

,환자번호,날짜,CC,약,장치,습관,찜질,"마사지, 스트레칭",PI,CMO,MMO,Cap.pal,M.pal,Noise,Loading,Occlusion,OJ/OB,Class,Midline Shift,Deviation,CR-CO,Tongue ridging,Mucosal ridging,Ultrasono,Rt,Lt,치료계획,End feel,CC_location,CC_pain_type,CC_painUncomp_desc_jaw,CC_disable_desc_jaw,CC_muscle_joint_desc_stress,CC_dentalHistory_desc,CC_clinic_history_desc,CC_factor_habbit,CC_treat_plan,CC_severity,CC_duration,약_medication_type,약_frequency,약_duration,약_compliance,장치_device_type,장치_usage_pattern,장치_duration,장치_compliance,습관_habit_type,습관_frequency,습관_awareness,습관_improvement,찜질_status,찜질_frequency,찜질_duration,찜질_method,"마사지, 스트레칭_type","마사지, 스트레칭_frequency","마사지, 스트레칭_duration","마사지, 스트레칭_method",CMO_before,CMO_after,MMO_before,MMO_after,deviation_pattern_type,deviation_direction,deviation_intensity,Cap.pal_Pain_Intensity,Cap.pal_Pain_Direction,Cap.pal_Pain_Situation,M.pal_Pain_Intensity,M.pal_Pain_Direction,M.pal_Pain_Situation,Noise_Code,Noise_Direction,Noise_Intensity,Noise_Situation,Occlusion_lt_number,Occlusion_rt_number,Occlusion_lt_Intensity,Occlusion_rt_Intensity,oj,ob,Midline_Shift_Jaw,Midline_Shift_Direction_x,Midline_Shift_Direction_y,Midline_Shift_Amount,CRCO_Direction_x,CRCO_Direction_y,CRCO_Amount,Tongue_ridging_Intensity,Mucosal_ridging_Intensity,Rt_before,Rt_after,Lt_before,Lt_after,Next_Visit_Days,CC_vas,dif_MMO_CMO,첫방문_CMO_before,CMO_before_일일성장률,첫방문_CC_vas,CC_vas_일일성장률,첫방문_MMO_before,MMO_before_일일성장률,첫방문_dif_MMO_CMO,dif_MMO_CMO_일일성장률,cluster_0_prob,cluster_1_prob,cluster_2_prob,cluster_3_prob,cluster_4_prob
5093,2204-105,2022-04-19,양쪽턱관절불편최근 1년 전부터 입이 안벌어져요 자주 빠져요 이사온지 얼마 안되서 턱...,Unknown,Unknown,Unknown,Unknown,Unknown,"하악 전치 교모양쪽 3,4번 실금12345678 12345678Dr.남윤진료측두하악...",22mm --> mm after spray and stretch,28mm --> 57mm after spray and stretch,Unknown,"both M) tenderpoint, 통증은 없음","Rt+ click, 고착해소 후 Lt+ popping",Unknown,Unknown,2/2,3,하악 오른쪽으로 1mm,왼쪽으로 L,Unknown,+,+,Unknown,1.13 -> 1.66,1.1-> 1.57,APS 한두달 후 SS,soft,양쪽턱관절,불편,입이 안벌어지고 자주 빠짐,턱이 걸려서 안벌어지는 느낌,스트레스는 없는데 작은 일에도 예민,Unknown,Unknown,Unknown,Unknown,5.0,10.0,Unknown,Unknown,NaN,Unknown,Unknown,Unknown,2.0,Unknown,치아 접촉 줄이기,Unknown,Unknown,Unknown,Unknown,Unknown,20.0,Unknown,Unknown,Unknown,10.0,Unknown,22.0,Unknown,28.0,57,L,left,normal,0,unspecified,Unknown,0,both,Unknown,Popping,both,0,고착 시,0,0,0,0,2.0,2.0,하악,오른쪽,Unknown,1.0,Unknown,Unknown,Unknown,1,1,1.13,1.66,1.13,1.66,60.0,6.0,6.0,22.0,0.000000,6.0,0.000000,28.0,0.000000,6.0,0.0,0.201422,0.191991,0.193403,0.210548,0.202637
5094,2204-105,2022-04-26,"구강내과#2물리치료 , APS del증상 : 자주 빠졌어요 음식먹기 불편했어요, 특...",Unknown,Unknown,Unknown,Unknown,Unknown,"하악 전치 교모양쪽 3,4번 실금- 개구량 좋아지셨으나 관절잡음 불편함 느끼신는 ...",22mm --> mm after spray and stretch,28mm --> mm after spray and stretch,Unknown,"both M) tenderpoint, 통증은 없음","Rt+ click, 고착해소 후 Lt+ popping",Unknown,Unknown,2/2,3,하악 오른쪽으로 1mm,왼쪽으로 L,Unknown,+,+,Unknown,1.13 -> 1.66,1.1-> 1.57,"물리치료 , 장치 ck [1주후]",soft,왼쪽 턱,통증,Unknown,Unknown,Unknown,Unknown,"3개월 전 증상 시작, 과거 외상 이력","이악물지않으려고 했고, 질긴음식안먹었어요",Unknown,5.0,10.0,Unknown,Unknown,NaN,Unknown,Unknown,Unknown,2.0,Unknown,치아 접촉 줄이기,Unknown,Unknown,Unknown,Unknown,Unknown,20.0,Unknown,Unknown,Unknown,10.0,Unknown,22.0,Unknown,28.0,Unknown,L,left,normal,0,unspecified,Unknown,0,both,Unknown,Popping,both,0,고착 시,0,0,0,0,2.0,2.0,하악,오른쪽,Unknown,1.0,Unknown,Unknown,Unknown,1,1,1.13,1.66,1.13,1.66,7.0,6.0,6.0,22.0,0.000000,6.0,0.000000,28.0,0.000000,6.0,0.0,0.201218,0.199154,0.200507,0.190965,0.208155
5095,2204-105,2022-05-12,"구강내과#3물리치료 , 장치 ck증상 : 입벌어지는 것 좋아졌다가 음식을 먹으면 불...",Unknown,Unknown,Unknown,Unknown,Unknown,"하악 전치 교모양쪽 3,4번 실금12345678 12345678Dr.남윤진료TMJ자...",54mm --> mm after spray and stretch,54mm --> mm after spray and stretch,Unknown,"both M) tenderpoint, 통증은 없음",both click,Unknown,Unknown,2/2,3,하악 오른쪽으로 1mm,왼쪽으로 L,Unknown,+,+,Unknown,1.13 -> 1.66,1.1-> 1.57,Unknown,soft,왼쪽 턱,통증,딱 소리,먹는 동안 걸렸다가 안걸렸다가,Unknown,Unknown,Unknown,Unknown,Unknown,5.0,10.0,Unknown,Unknown,NaN,Unknown,Unknown,Unknown,2.0,Unknown,치아 접촉 줄이기,Unknown,Unknown,Unknown,Unknown,Unknow

In [59]:
df.dif_MMO_CMO.value_counts()

dif_MMO_CMO
0.0     13925
2.0      2541
3.0      1941
5.0      1890
4.0      1109
1.0      1076
8.0       727
7.0       725
6.0       654
10.0      647
9.0       353
12.0      307
11.0      283
15.0      270
13.0      219
20.0      170
14.0      150
16.0      131
18.0      130
17.0      124
22.0       97
19.0       72
27.0       53
25.0       52
23.0       47
24.0       42
21.0       40
30.0       36
28.0       27
26.0       24
29.0        8
32.0        8
31.0        6
35.0        5
36.0        4
34.0        3
40.0        1
37.0        1
33.0        1
38.0        1
Name: count, dtype: int64

In [62]:
df_sp[['CC_location']].value_counts()

CC_location   
왼쪽 턱              1517
오른쪽 턱              221
양쪽 턱               180
왼쪽                 168
오른쪽                113
                  ... 
오른쪽 아래 사랑니           1
오른쪽 아래 맨끝에 어금니       1
오른쪽 아래 맨끝 뒷쪽         1
오른쪽 볼, 턱             1
혜 앞쪽                 1
Name: count, Length: 491, dtype: int64

In [ ]:
# def main():
#     """전체 파이프라인 실행 및 테스트"""
#     import os
#     import pandas as pd
#     from datetime import datetime
    
#     # 1. 데이터 로드
#     print("데이터 로드 중...")
#     df = pd.read_hdf('/Users/nam-yeong/git/prj_centum/gpt_word/final_result/preprocessed_final_df.h5', key='df')
#     print(f"로드된 데이터 크기: {df.shape}")
    
#     # 2. API 키 설정
#     api_key = os.environ.get('OPENAI_API_KEY')
#     if not api_key:
#         api_key = input("OpenAI API 키를 입력하세요: ")
    
#     # 3. 전체 파이프라인 실행
#     prediction_system = run_complete_predictive_pipeline(
#         df, api_key, n_clusters=5, temperature=0.1, recovery_threshold=70
#     )
    
#     # 4. 예측 시스템 테스트
#     # 4.1 임의의 환자 선택
#     sample_patient = df['환자번호'].sample(1).iloc[0]
#     run_example_prediction(prediction_system, sample_patient)
    
#     # 4.2 (선택적) 대화형 대시보드 생성
#     create_interactive_dashboard(prediction_system, df)
    
#     # 5. 결과 저장
#     timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
#     result_path = f'/Users/nam-yeong/git/prj_centum/gpt_word/final_result/prediction_model_{timestamp}.pkl'
    
#     import pickle
#     with open(result_path, 'wb') as f:
#         # 저장이 불가능한 콜백 함수 등 제외
#         save_dict = {k: v for k, v in prediction_system.items() 
#                     if k not in ['predict_recovery']}
#         pickle.dump(save_dict, f)
    
#     print(f"\n예측 모델이 '{result_path}'에 저장되었습니다.")
#     print("\n실행 완료!")

# if __name__ == "__main__":
    # embedding_batch_size = 100
    
    # main(embedding_batch_size)